# Q1b — Word2Vec embeddings → two-layer MLP

**W&B run:** _filled in after first run_

This notebook trains two word2vec checkpoints (1 epoch and 20 epochs) from byte-identical initial weights, compares their embedding space with t-SNE, then trains the project MLP on mean-pooled 20-epoch vectors.


In [1]:
%load_ext autoreload
%autoreload 2

import os
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import wandb
from torch.utils.data import DataLoader, TensorDataset

from nlp_project import SEED, set_seed
from nlp_project.data import load_20ng, preprocess, train_val_split
from nlp_project.embeddings import train_word2vec, mean_pool
from nlp_project.eval import evaluate, plot_confusion
from nlp_project.model import MLP
from nlp_project.train import train as train_loop
from nlp_project.viz import plot_word_neighborhood

set_seed()
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path("../models"); MODEL_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [2]:
# Load and preprocess.
# For word2vec we KEEP stopwords — co-occurrence with frequent function
# words helps the embedding signal (see spec §8).
train_docs, train_labels, test_docs, test_labels, label_names = load_20ng(remove=True)
train_tokens_w2v = preprocess(train_docs, drop_stopwords=False)
test_tokens_w2v = preprocess(test_docs, drop_stopwords=False)


In [3]:
# Train both word2vec checkpoints.
m1 = train_word2vec(train_tokens_w2v, epochs=1, vector_size=100, seed=SEED)
m1.wv.save(str(MODEL_DIR / "w2v_epoch1.kv"))

m20 = train_word2vec(train_tokens_w2v, epochs=20, vector_size=100, seed=SEED)
m20.wv.save(str(MODEL_DIR / "w2v_epoch20.kv"))

print(f"vocab(epoch=1):  {len(m1.wv)}")
print(f"vocab(epoch=20): {len(m20.wv)}")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

vocab(epoch=1):  19306
vocab(epoch=20): 19306


In [4]:
# t-SNE comparison: 500 most frequent tokens common to both checkpoints.
counter = Counter()
for toks in train_tokens_w2v:
    counter.update(toks)

common = [w for w, _ in counter.most_common(2000)
          if w in m1.wv.key_to_index and w in m20.wv.key_to_index][:500]
print(f"plotting {len(common)} tokens")

plot_word_neighborhood(
    {"word2vec — 1 epoch": m1, "word2vec — 20 epochs": m20},
    common,
    save_path=FIG_DIR / "w2v_tsne_epoch1_vs_epoch20.png",
)


plotting 500 tokens


In [5]:
# Build mean-pooled doc vectors with the 20-epoch model.
X_train_full = mean_pool(train_tokens_w2v, m20)
X_test = mean_pool(test_tokens_w2v, m20)
y_train_full = train_labels
y_test = test_labels
print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")


X_train_full: (11314, 100), X_test: (7532, 100)


In [7]:
# Train/val split + DataLoaders.
X_train, y_train, X_val, y_val = train_val_split(
    list(X_train_full), y_train_full, val_frac=0.1, seed=SEED,
)
X_train, X_val = np.asarray(X_train), np.asarray(X_val)

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, 64, shuffle=True)
val_loader = make_loader(X_val, y_val, 64, shuffle=False)
test_loader = make_loader(X_test, y_test, 64, shuffle=False)


In [8]:
# W&B run + train.
run = wandb.init(
    project="hslu-nalapro",
    name="q1b-word2vec-meanpool",
    config={"vectorizer": "word2vec-meanpool", "vector_size": 100,
            "w2v_epochs": 20, "hidden_dim": 256, "dropout": 0.3,
            "lr": 1e-3, "batch_size": 64},
)

model = MLP(in_dim=100, hidden_dim=256, num_classes=20, dropout=0.3)
history = train_loop(
    model, train_loader, val_loader,
    epochs=50, lr=1e-3, device=DEVICE, wandb_run=run, patience=5,
)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/dan/.netrc.
wandb: Currently logged in as: danwwaititu (danwwaititu-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [9]:
# Evaluate on the test set, log, finish run.
metrics = evaluate(model, test_loader, label_names, device=DEVICE)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test macro-F1: {metrics['macro_f1']:.4f}")
plot_confusion(
    metrics["confusion_matrix"], label_names,
    save_path=FIG_DIR / "confusion_matrix_q1b.png",
    title="Q1b — word2vec mean-pool",
)
run.log({"test_accuracy": metrics["accuracy"], "test_macro_f1": metrics["macro_f1"]})
run.finish()


test accuracy: 0.6321
test macro-F1: 0.6098


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
test_accuracy,▁
test_macro_f1,▁
train_acc,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇██████████████
train_loss,█▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▅▆▆▆▇▇▇▇▇▇▇▇█▇▇██████████████████
val_loss,█▅▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,35
test_accuracy,0.6321
test_macro_f1,0.60981
train_acc,0.70242


## Notes for the report

- After 20 epochs the t-SNE plot shows visibly clustered semantic neighbourhoods (e.g. "car/engine/wheel" vs "god/jesus/bible") that are absent at 1 epoch — that's the visualization the spec asks for.
- Mean-pool baseline: this is the floor that 1c (TF-IDF) and 1d (mean+max-pool) need to beat.
